# Free Keyword Suggestion / Autocomplete Endpoint Tester — v2 Merged

Colab notebook that tests GET-based keyword suggestion/autocomplete endpoints and returns a final table of working/not-working/skipped endpoints with example results.

This v2 merges the original list with the newer verified/provider-surfaced and Firefox/browser-config suggestions: Datamuse, Startpage, Qwant, current Google/Bing/DuckDuckGo variants, regional browser-config engines, and keyed Brave official autosuggest.


In [ ]:
!pip -q install requests pandas beautifulsoup4 lxml tabulate


In [ ]:
# =========================
# CONFIG
# =========================

QUERY = "salesforce ai"
LANG = "en"
COUNTRY = "US"
MARKET = "en-us"

MAX_RESULTS_SHOWN = 10
REQUEST_TIMEOUT_SECONDS = 15
DELAY_BETWEEN_REQUESTS_SECONDS = 0.75

APP_USER_AGENT = "keyword-endpoint-tester-v2/1.0 (Colab; contact: replace-me@example.com)"

RUN_EXPERIMENTAL_ENDPOINTS = True
RUN_REGIONAL_ENDPOINTS = True
RUN_ECOMMERCE_ALIAS_VARIANTS = True

SAVE_TO_GOOGLE_DRIVE = False
GOOGLE_DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/keyword_endpoint_tests"
LOCAL_OUTPUT_DIR = "/content/keyword_endpoint_tests"


In [ ]:
# =========================
# IMPORTS / SECRETS / PARSERS / ENDPOINTS
# =========================

from __future__ import annotations

import os, re, json, time, html, pathlib
import datetime as dt
import xml.etree.ElementTree as ET
from dataclasses import dataclass
from typing import Callable, Any, Optional
from urllib.parse import quote_plus

import requests
import pandas as pd

try:
    from bs4 import BeautifulSoup
except Exception:
    BeautifulSoup = None

def get_secret(name: str, default: str = "") -> str:
    val = os.getenv(name)
    if val:
        return val
    try:
        from google.colab import userdata
        return userdata.get(name) or default
    except Exception:
        return default

API_KEYS = {
    "BRAVE_SEARCH_API_KEY": get_secret("BRAVE_SEARCH_API_KEY"),
    "OPENCAGE_API_KEY": get_secret("OPENCAGE_API_KEY"),
    "LASTFM_API_KEY": get_secret("LASTFM_API_KEY"),
    "RAWG_API_KEY": get_secret("RAWG_API_KEY"),
}

session = requests.Session()
session.headers.update({
    "User-Agent": APP_USER_AGENT,
    "Accept": "application/json, text/plain, text/html, application/xml, text/xml, */*",
    "Accept-Language": f"{LANG},{LANG}-{COUNTRY};q=0.9,en-US;q=0.8,en;q=0.7",
})

# ---------- parser helpers ----------

def clean_suggestions(items: Any, max_items: int = MAX_RESULTS_SHOWN) -> list[str]:
    flat: list[str] = []
    preferred = [
        "phrase", "value", "word", "key", "name", "title", "trackName", "artistName",
        "collectionName", "display_name", "label", "term", "query", "suggestion",
        "text", "keyword", "display"
    ]

    def walk(x: Any):
        if x is None:
            return
        if isinstance(x, str):
            s = html.unescape(x)
            s = re.sub(r"<[^>]+>", " ", s)
            s = re.sub(r"\s+", " ", s).strip()
            if s and 1 < len(s) <= 180:
                flat.append(s)
            return
        if isinstance(x, (int, float, bool)):
            return
        if isinstance(x, dict):
            for k in preferred:
                if k in x:
                    walk(x[k])
            for k, v in x.items():
                if k not in preferred and isinstance(v, (dict, list, tuple)):
                    walk(v)
            return
        if isinstance(x, (list, tuple, set)):
            for i in x:
                walk(i)

    walk(items)
    out, seen = [], set()
    junk_prefixes = ("http://", "https://", "{", "[", "javascript:")
    for s in flat:
        s = s.strip(" \t\r\n\"'")
        if not s or s.lower() in {"true", "false", "null", "none"}:
            continue
        if s.lower().startswith(junk_prefixes):
            continue
        if len(s.split()) > 20:
            continue
        key = s.lower()
        if key not in seen:
            out.append(s)
            seen.add(key)
        if len(out) >= max_items:
            break
    return out

def _extract_jsonish(text: str) -> Any:
    text = text.strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    m = re.search(r"^[\w.$]+\((.*)\)\s*;?$", text, flags=re.S)
    if m:
        try:
            return json.loads(m.group(1).strip())
        except Exception:
            pass
    m = re.search(r"(\[.*\]|\{.*\})", text, flags=re.S)
    if m:
        try:
            return json.loads(m.group(1).strip())
        except Exception:
            pass
    return text

def json_or_text(response: requests.Response) -> Any:
    try:
        return response.json()
    except Exception:
        return _extract_jsonish(response.text)

def parse_google_like(response):
    data = json_or_text(response)
    if isinstance(data, list):
        if len(data) >= 2 and isinstance(data[1], list):
            return clean_suggestions(data[1])
        return clean_suggestions(data)
    return clean_suggestions(data)

def parse_xml_suggestions(response):
    text = response.text
    try:
        root = ET.fromstring(text)
        vals = []
        for node in root.iter():
            if node.tag.lower().endswith("suggestion"):
                v = node.attrib.get("data")
                if v:
                    vals.append(v)
        return clean_suggestions(vals)
    except Exception:
        pass
    return clean_suggestions(re.findall(r'data="([^"]+)"', text))

def parse_bing_osjson(response):
    data = json_or_text(response)
    if isinstance(data, list) and len(data) >= 2:
        return clean_suggestions(data[1])
    return clean_suggestions(data)

def parse_html_fragment(response):
    text = response.text
    if BeautifulSoup:
        soup = BeautifulSoup(text, "html.parser")
        vals = []
        for selector in [".sa_tm_text", ".sa_tm", ".match_name", ".title", "li", "a", "span", "div"]:
            vals.extend([n.get_text(" ", strip=True) for n in soup.select(selector)])
            vals = [v for v in vals if v]
            if vals:
                return clean_suggestions(vals)
    return clean_suggestions(re.findall(r">([^<>]{2,120})<", text))

def parse_ddg(response):
    data = json_or_text(response)
    if isinstance(data, list):
        return clean_suggestions([i.get("phrase") if isinstance(i, dict) else i for i in data])
    return clean_suggestions(data)

def parse_datamuse(response):
    data = json_or_text(response)
    if isinstance(data, list):
        return clean_suggestions([i.get("word") if isinstance(i, dict) else i for i in data])
    return clean_suggestions(data)

def parse_yahoo(response):
    data = json_or_text(response)
    if isinstance(data, dict):
        if "r" in data:
            return clean_suggestions([i.get("k") for i in data.get("r", []) if isinstance(i, dict)])
        gossip = data.get("gossip") or {}
        if isinstance(gossip, dict):
            return clean_suggestions(gossip.get("results") or gossip.get("result") or [])
    return clean_suggestions(data)

def parse_amazon_api_2017(response):
    data = json_or_text(response)
    if isinstance(data, dict):
        return clean_suggestions([s.get("value") for s in data.get("suggestions", []) if isinstance(s, dict)])
    return clean_suggestions(data)

def parse_amazon_legacy(response):
    data = json_or_text(response)
    if isinstance(data, list) and len(data) >= 2:
        return clean_suggestions(data[1])
    return clean_suggestions(data)

def parse_ebay(response):
    data = json_or_text(response)
    if isinstance(data, dict):
        return clean_suggestions(data.get("res", {}).get("sug", []))
    return clean_suggestions(data)

def parse_wiki(response):
    data = json_or_text(response)
    if isinstance(data, list) and len(data) >= 2:
        return clean_suggestions(data[1])
    return clean_suggestions(data)

def parse_itunes(response):
    data = json_or_text(response)
    if isinstance(data, dict):
        vals = []
        for i in data.get("results", []):
            if isinstance(i, dict):
                vals.append(i.get("trackName") or i.get("artistName") or i.get("collectionName") or i.get("trackCensoredName"))
        return clean_suggestions(vals)
    return clean_suggestions(data)

def parse_nominatim(response):
    data = json_or_text(response)
    if isinstance(data, list):
        return clean_suggestions([i.get("display_name") or i.get("name") for i in data if isinstance(i, dict)])
    return clean_suggestions(data)

def parse_photon(response):
    data = json_or_text(response)
    if isinstance(data, dict):
        vals = []
        for f in data.get("features", []):
            props = f.get("properties", {}) if isinstance(f, dict) else {}
            parts = [props.get("name"), props.get("city"), props.get("country")]
            s = ", ".join([p for p in parts if p])
            if s:
                vals.append(s)
        return clean_suggestions(vals)
    return clean_suggestions(data)

def parse_musicbrainz(response):
    data = json_or_text(response)
    if isinstance(data, dict):
        for k in ["artists", "releases", "recordings"]:
            if k in data:
                return clean_suggestions([i.get("name") or i.get("title") for i in data[k] if isinstance(i, dict)])
    return clean_suggestions(data)

def parse_openlibrary_authors(response):
    data = json_or_text(response)
    if isinstance(data, dict):
        return clean_suggestions([i.get("name") for i in data.get("docs", []) if isinstance(i, dict)])
    return clean_suggestions(data)

def parse_openlibrary_books(response):
    data = json_or_text(response)
    if isinstance(data, dict):
        vals = []
        for i in data.get("docs", []):
            if not isinstance(i, dict):
                continue
            title = i.get("title")
            authors = i.get("author_name") or []
            if title and authors:
                vals.append(f"{title} — {', '.join(authors[:2])}")
            elif title:
                vals.append(title)
        return clean_suggestions(vals)
    return clean_suggestions(data)

def parse_generic(response):
    return clean_suggestions(json_or_text(response))

# ---------- endpoint definitions ----------

@dataclass
class Endpoint:
    category: str
    name: str
    url_template: str
    parser: Callable[[requests.Response], list[str]]
    stability: str = "unofficial"
    source_batch: str = "original"
    requires_key: Optional[str] = None
    enabled: bool = True
    notes: str = ""
    headers: Optional[dict[str, str]] = None

def cp_for_query(q: str) -> int:
    return max(1, min(len(q), 50))

NEW_ENTRIES_ADDED = [
    "Datamuse /sug",
    "Startpage osuggestions",
    "Google current Firefox config",
    "Bing current Firefox config",
    "DuckDuckGo current Firefox config",
    "Qwant Firefox/OpenSearch suggest",
    "Baidu Firefox/OpenSearch suggest",
    "Naver Firefox/OpenSearch suggest",
    "Daum Firefox/OpenSearch suggest",
    "Coccoc Firefox suggest",
    "1&1 suggest",
    "web.de suggest",
    "mail.com suggest",
    "GMX DE suggest",
    "Brave official Autosuggest API with token",
    "Amazon US alias variants: computers, electronics, books, music, software",
    "Explicit Wikipedia language variants: de, fr, es, ja",
]

ENDPOINTS: list[Endpoint] = [
    # v2 verified/provider-surfaced/browser-config additions
    Endpoint("General Web", "Google current Firefox config", "https://www.google.com/complete/search?client=firefox&channel=fen&q={query}", parse_google_like, "browser_config", "v2_merged"),
    Endpoint("General Web", "Bing current Firefox config", "https://www.bing.com/osjson.aspx?query={query}&form=OSDJAS", parse_bing_osjson, "browser_config", "v2_merged"),
    Endpoint("General Web", "DuckDuckGo current Firefox config", "https://ac.duckduckgo.com/ac/?q={query}&type=list", parse_ddg, "browser_config", "v2_merged"),
    Endpoint("General Web", "Startpage osuggestions", "https://www.startpage.com/osuggestions?q={query}", parse_generic, "provider_help", "v2_merged"),
    Endpoint("General Web", "Datamuse sug", "https://api.datamuse.com/sug?s={query}", parse_datamuse, "official", "v2_merged"),
    Endpoint("General Web", "Qwant Firefox/OpenSearch suggest", "https://api.qwant.com/api/suggest/?client=opensearch&q={query}", parse_google_like, "browser_config", "v2_merged"),
    Endpoint("General Web", "Brave official Autosuggest API", "https://api.search.brave.com/res/v1/suggest/search?q={query}", parse_generic, "official_keyed", "v2_merged", requires_key="BRAVE_SEARCH_API_KEY", headers={"X-Subscription-Token": "{BRAVE_SEARCH_API_KEY}"}),

    # regional/browser-config additions
    Endpoint("Regional Search", "Baidu OpenSearch suggest", "https://www.baidu.com/su?wd={query}&ie=utf-8&action=opensearch", parse_google_like, "browser_config", "v2_merged", enabled=RUN_REGIONAL_ENDPOINTS),
    Endpoint("Regional Search", "Naver OpenSearch suggest", "https://ac.search.naver.com/nx/ac?of=os&ie=utf-8&q={query}", parse_generic, "browser_config", "v2_merged", enabled=RUN_REGIONAL_ENDPOINTS),
    Endpoint("Regional Search", "Daum OpenSearch suggest", "https://suggest.search.daum.net/sushi/opensearch/pc?DA=JU2&q={query}", parse_google_like, "browser_config", "v2_merged", enabled=RUN_REGIONAL_ENDPOINTS),
    Endpoint("Regional Search", "Coccoc autocomplete", "https://coccoc.com/composer/autocomplete?of=b&s=ff&q={query}", parse_generic, "browser_config", "v2_merged", enabled=RUN_REGIONAL_ENDPOINTS),
    Endpoint("Regional Search", "1&1 suggest", "https://suggestplugin.ui-portal.de/s?brand=1und1&origin=br_splugin_ff_sg&q={query}", parse_generic, "browser_config", "v2_merged", enabled=RUN_REGIONAL_ENDPOINTS),
    Endpoint("Regional Search", "web.de suggest", "https://suggestplugin.ui-portal.de/s?brand=webde&origin=br_splugin_ff_sg&q={query}", parse_generic, "browser_config", "v2_merged", enabled=RUN_REGIONAL_ENDPOINTS),
    Endpoint("Regional Search", "mail.com suggest", "https://search.mail.com/SuggestSearch/s?brand=mailcom&origin=br_splugin_ff_sg&q={query}", parse_generic, "browser_config", "v2_merged", enabled=RUN_REGIONAL_ENDPOINTS),
    Endpoint("Regional Search", "GMX DE suggest", "https://suggestplugin.ui-portal.de/s?brand=gmx&origin=br_splugin_ff_sg&q={query}", parse_generic, "browser_config", "v2_merged", enabled=RUN_REGIONAL_ENDPOINTS),

    # original general web endpoints retained/refreshed
    Endpoint("General Web", "Google Firefox JSON legacy", "https://suggestqueries.google.com/complete/search?client=firefox&hl={lang}&q={query}", parse_google_like, "unofficial", "original"),
    Endpoint("General Web", "Google Chrome JSON", "https://www.google.com/complete/search?client=chrome&q={query}&hl={lang}", parse_google_like, "unofficial", "original"),
    Endpoint("General Web", "Google Toolbar XML", "http://google.com/complete/search?output=toolbar&gl={country}&q={query}", parse_xml_suggestions, "unofficial", "original"),
    Endpoint("General Web", "Google Chrome Omni", "https://www.google.com/complete/search?client=chrome-omni&gs_ri=chrome-ext&q={query}", parse_google_like, "unofficial", "original"),
    Endpoint("General Web", "Google News", "https://suggestqueries.google.com/complete/search?client=firefox&ds=n&q={query}", parse_google_like, "unofficial", "original"),
    Endpoint("General Web", "Google Shopping", "https://suggestqueries.google.com/complete/search?client=firefox&ds=sh&q={query}", parse_google_like, "unofficial", "original"),
    Endpoint("General Web", "Google Images", "https://suggestqueries.google.com/complete/search?client=firefox&ds=i&q={query}", parse_google_like, "unofficial", "original"),
    Endpoint("General Web", "Bing OS JSON legacy host", "https://api.bing.com/osjson.aspx?query={query}&language={lang}", parse_bing_osjson, "unofficial", "original"),
    Endpoint("General Web", "Bing qsonhs fragment", "https://www.bing.com/AS/Suggestions?pt=page.serp&mkt={market}&qry={query}&cp={cp}&cvid=1", parse_html_fragment, "unofficial", "original"),
    Endpoint("General Web", "DuckDuckGo original host", "https://duckduckgo.com/ac/?q={query}&type=list", parse_ddg, "unofficial", "original"),
    Endpoint("General Web", "Ecosia autocomplete", "https://ac.ecosia.org/autocomplete?type=list&q={query}", parse_google_like, "provider_help", "original_refreshed"),
    Endpoint("General Web", "Yahoo gossip", "https://search.yahoo.com/sugg/gossip/gossip-us-ura/?command={query}&output=json&nresults=20", parse_yahoo, "experimental", "original", enabled=RUN_EXPERIMENTAL_ENDPOINTS),
    Endpoint("General Web", "Brave legacy unauth endpoint", "https://search.brave.com/api/suggest?q={query}&rich=false&source=web", parse_generic, "experimental", "original", enabled=RUN_EXPERIMENTAL_ENDPOINTS),
    Endpoint("General Web", "Ask.com", "https://ss.ask.com/query?q={query}&li=ff&start=0&num=10", parse_generic, "experimental", "original", enabled=RUN_EXPERIMENTAL_ENDPOINTS),
    Endpoint("General Web", "Yandex", "https://suggest.yandex.com/suggest-ff.cgi?part={query}&uil=en&lr=0", parse_google_like, "unofficial", "original"),

    # video
    Endpoint("Video", "YouTube Firefox", "https://suggestqueries.google.com/complete/search?client=firefox&ds=yt&hl={lang}&q={query}", parse_google_like, "experimental", "original", enabled=RUN_EXPERIMENTAL_ENDPOINTS),
    Endpoint("Video", "YouTube alternate", "https://suggestqueries-clients6.youtube.com/complete/search?client=youtube&ds=yt&q={query}", parse_google_like, "experimental", "original", enabled=RUN_EXPERIMENTAL_ENDPOINTS),
    Endpoint("Video", "Vimeo public video search", "https://vimeo.com/api/v2/videos/search.json?query={query}&per_page=10", parse_generic, "experimental", "original", enabled=RUN_EXPERIMENTAL_ENDPOINTS),

    # ecommerce
    Endpoint("E-Commerce", "Amazon US API 2017 all", "https://completion.amazon.com/api/2017/suggestions?mid=ATVPDKIKX0DER&alias=aps&prefix={query}", parse_amazon_api_2017, "unofficial", "original"),
    Endpoint("E-Commerce", "Amazon US API 2017 computers", "https://completion.amazon.com/api/2017/suggestions?mid=ATVPDKIKX0DER&alias=computers&prefix={query}", parse_amazon_api_2017, "unofficial", "v2_merged", enabled=RUN_ECOMMERCE_ALIAS_VARIANTS),
    Endpoint("E-Commerce", "Amazon US API 2017 electronics", "https://completion.amazon.com/api/2017/suggestions?mid=ATVPDKIKX0DER&alias=electronics&prefix={query}", parse_amazon_api_2017, "unofficial", "v2_merged", enabled=RUN_ECOMMERCE_ALIAS_VARIANTS),
    Endpoint("E-Commerce", "Amazon US API 2017 books", "https://completion.amazon.com/api/2017/suggestions?mid=ATVPDKIKX0DER&alias=books&prefix={query}", parse_amazon_api_2017, "unofficial", "v2_merged", enabled=RUN_ECOMMERCE_ALIAS_VARIANTS),
    Endpoint("E-Commerce", "Amazon US API 2017 music", "https://completion.amazon.com/api/2017/suggestions?mid=ATVPDKIKX0DER&alias=music&prefix={query}", parse_amazon_api_2017, "unofficial", "v2_merged", enabled=RUN_ECOMMERCE_ALIAS_VARIANTS),
    Endpoint("E-Commerce", "Amazon US API 2017 software", "https://completion.amazon.com/api/2017/suggestions?mid=ATVPDKIKX0DER&alias=software&prefix={query}", parse_amazon_api_2017, "unofficial", "v2_merged", enabled=RUN_ECOMMERCE_ALIAS_VARIANTS),
    Endpoint("E-Commerce", "Amazon legacy US", "http://completion.amazon.com/search/complete?search-alias=aps&client=amazon-search-ui&mkt=1&q={query}", parse_amazon_legacy, "unofficial", "original"),
    Endpoint("E-Commerce", "Amazon UK API 2017", "https://completion.amazon.co.uk/api/2017/suggestions?mid=A1F83G8C2ARO7P&alias=aps&prefix={query}", parse_amazon_api_2017, "unofficial", "original"),
    Endpoint("E-Commerce", "Amazon DE API 2017", "https://completion.amazon.de/api/2017/suggestions?mid=A1PA6795UKMFR9&alias=aps&prefix={query}", parse_amazon_api_2017, "unofficial", "original"),
    Endpoint("E-Commerce", "eBay autosuggest", "https://autosug.ebay.com/autosug?sId=0&kwd={query}", parse_ebay, "unofficial", "original"),

    # knowledge/reference
    Endpoint("Knowledge", "Wikipedia OpenSearch EN", "https://en.wikipedia.org/w/api.php?action=opensearch&search={query}&limit=10&namespace=0&format=json", parse_wiki, "official", "original"),
    Endpoint("Knowledge", "Wikipedia OpenSearch LANG config", "https://{lang}.wikipedia.org/w/api.php?action=opensearch&search={query}&limit=10&format=json", parse_wiki, "official", "original"),
    Endpoint("Knowledge", "Wikipedia OpenSearch DE", "https://de.wikipedia.org/w/api.php?action=opensearch&search={query}&limit=10&format=json", parse_wiki, "official", "v2_merged"),
    Endpoint("Knowledge", "Wikipedia OpenSearch FR", "https://fr.wikipedia.org/w/api.php?action=opensearch&search={query}&limit=10&format=json", parse_wiki, "official", "v2_merged"),
    Endpoint("Knowledge", "Wikipedia OpenSearch ES", "https://es.wikipedia.org/w/api.php?action=opensearch&search={query}&limit=10&format=json", parse_wiki, "official", "v2_merged"),
    Endpoint("Knowledge", "Wikipedia OpenSearch JA", "https://ja.wikipedia.org/w/api.php?action=opensearch&search={query}&limit=10&format=json", parse_wiki, "official", "v2_merged"),
    Endpoint("Knowledge", "Wiktionary OpenSearch", "https://en.wiktionary.org/w/api.php?action=opensearch&search={query}&limit=10&format=json", parse_wiki, "official", "original"),
    Endpoint("Knowledge", "Wikimedia Commons OpenSearch", "https://commons.wikimedia.org/w/api.php?action=opensearch&search={query}&limit=10&format=json", parse_wiki, "official", "original"),

    # places
    Endpoint("Places", "Nominatim OpenStreetMap", "https://nominatim.openstreetmap.org/search?q={query}&format=json&limit=10&addressdetails=1", parse_nominatim, "official", "original", notes="Requires descriptive User-Agent and low volume."),
    Endpoint("Places", "Photon Komoot", "https://photon.komoot.io/api/?q={query}&limit=10&lang=en", parse_photon, "official", "original"),
    Endpoint("Places", "OpenCage Geocoding", "https://api.opencagedata.com/geocode/v1/json?q={query}&key={OPENCAGE_API_KEY}&limit=10", parse_generic, "official_keyed", "original", requires_key="OPENCAGE_API_KEY"),
    Endpoint("Places", "UK Government Search", "https://www.gov.uk/api/search.json?q={query}&suggest=autocomplete&count=1", parse_generic, "official", "original"),

    # music/media
    Endpoint("Music / Media", "MusicBrainz artist", "https://musicbrainz.org/ws/2/artist?query={query}&fmt=json&limit=10", parse_musicbrainz, "official", "original"),
    Endpoint("Music / Media", "MusicBrainz release", "https://musicbrainz.org/ws/2/release?query={query}&fmt=json&limit=10", parse_musicbrainz, "official", "original"),
    Endpoint("Music / Media", "MusicBrainz recording", "https://musicbrainz.org/ws/2/recording?query={query}&fmt=json&limit=10", parse_musicbrainz, "official", "original"),
    Endpoint("Music / Media", "Last.fm artist search", "https://ws.audioscrobbler.com/2.0/?method=artist.search&artist={query}&api_key={LASTFM_API_KEY}&format=json&limit=10", parse_generic, "official_keyed", "original", requires_key="LASTFM_API_KEY"),
    Endpoint("Music / Media", "iTunes music artists", "https://itunes.apple.com/search?term={query}&limit=10&entity=musicArtist", parse_itunes, "official", "original"),
    Endpoint("Music / Media", "iTunes software", "https://itunes.apple.com/search?term={query}&limit=10&entity=software", parse_itunes, "official", "original"),
    Endpoint("Music / Media", "iTunes movies", "https://itunes.apple.com/search?term={query}&limit=10&entity=movie", parse_itunes, "official", "original"),

    # gaming/apps
    Endpoint("Gaming / Apps", "Steam suggest", "https://store.steampowered.com/search/suggest?term={query}&f=games&cc={country}&l=english", parse_html_fragment, "unofficial", "original"),
    Endpoint("Gaming / Apps", "RAWG games", "https://api.rawg.io/api/games?key={RAWG_API_KEY}&search={query}&page_size=10", parse_generic, "official_keyed", "original", requires_key="RAWG_API_KEY"),

    # scientific/specialty
    Endpoint("Scientific / Specialty", "PubMed NCBI esuggest", "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esuggest.fcgi?db=pubmed&q={query}&retmode=json", parse_generic, "official", "original"),
    Endpoint("Scientific / Specialty", "NASA Earthdata CMR autocomplete", "https://cmr.earthdata.nasa.gov/search/autocomplete?q={query}", parse_generic, "official", "original"),
    Endpoint("Scientific / Specialty", "Open Library authors", "https://openlibrary.org/search/authors.json?q={query}&limit=10", parse_openlibrary_authors, "official", "original"),
    Endpoint("Scientific / Specialty", "Open Library books", "https://openlibrary.org/search.json?q={query}&limit=10&fields=title,author_name", parse_openlibrary_books, "official", "original"),
]

print(f"Total endpoint definitions: {len(ENDPOINTS)}")
print(f"v2 merged additions/variants: {sum(1 for e in ENDPOINTS if e.source_batch == 'v2_merged')}")
print("\nNew entries added:")
for x in NEW_ENTRIES_ADDED:
    print("-", x)


In [ ]:
# =========================
# RUNNER
# =========================

def format_headers(headers: Optional[dict[str, str]]) -> dict[str, str]:
    final = dict(headers or {})
    for k, v in list(final.items()):
        if isinstance(v, str) and v.startswith("{") and v.endswith("}"):
            final[k] = API_KEYS.get(v.strip("{}"), "")
    final.setdefault("User-Agent", APP_USER_AGENT)
    return final

def build_url(template: str, query: str) -> str:
    values = {
        "query": quote_plus(query),
        "lang": quote_plus(LANG),
        "country": quote_plus(COUNTRY),
        "market": quote_plus(MARKET),
        "cp": cp_for_query(query),
        **{k: quote_plus(v) for k, v in API_KEYS.items()},
    }
    return template.format(**values)

def test_endpoint(endpoint: Endpoint, query: str = QUERY) -> dict[str, Any]:
    started = time.perf_counter()

    if not endpoint.enabled:
        return {
            "category": endpoint.category, "endpoint": endpoint.name, "stability": endpoint.stability,
            "source_batch": endpoint.source_batch, "status": "DISABLED_EXPERIMENTAL", "working": False,
            "http_status": None, "latency_ms": None, "result_count": 0, "examples": [],
            "examples_preview": "", "url": endpoint.url_template, "error": "Endpoint disabled by feature flag.",
            "notes": endpoint.notes,
        }

    if endpoint.requires_key and not API_KEYS.get(endpoint.requires_key):
        return {
            "category": endpoint.category, "endpoint": endpoint.name, "stability": endpoint.stability,
            "source_batch": endpoint.source_batch, "status": "SKIPPED_KEY_REQUIRED", "working": False,
            "http_status": None, "latency_ms": None, "result_count": 0, "examples": [],
            "examples_preview": "", "url": endpoint.url_template,
            "error": f"Missing {endpoint.requires_key}. Add it as an env var or Colab Secret.",
            "notes": endpoint.notes,
        }

    url = build_url(endpoint.url_template, query)
    try:
        response = session.get(url, headers=format_headers(endpoint.headers), timeout=REQUEST_TIMEOUT_SECONDS, allow_redirects=True)
        latency_ms = round((time.perf_counter() - started) * 1000, 1)
        parse_error = ""
        try:
            examples = endpoint.parser(response)
        except Exception as parser_exc:
            parse_error = f"Parser error: {type(parser_exc).__name__}: {parser_exc}"
            try:
                examples = parse_generic(response)
            except Exception:
                examples = []

        ok_http = 200 <= response.status_code < 300
        ok_results = len(examples) > 0

        if ok_http and ok_results:
            status, working, error = "WORKING", True, parse_error
        elif ok_http and not ok_results:
            sample = response.text[:300].replace("\n", " ")
            status, working, error = "HTTP_OK_NO_RESULTS", False, parse_error or f"No parsed results. Response sample: {sample}"
        else:
            sample = response.text[:300].replace("\n", " ")
            status, working, error = "HTTP_ERROR", False, f"HTTP {response.status_code}. {parse_error} Response sample: {sample}"

        return {
            "category": endpoint.category, "endpoint": endpoint.name, "stability": endpoint.stability,
            "source_batch": endpoint.source_batch, "status": status, "working": working,
            "http_status": response.status_code, "latency_ms": latency_ms, "result_count": len(examples),
            "examples": examples[:MAX_RESULTS_SHOWN], "examples_preview": " | ".join(examples[:5]),
            "url": url, "error": error, "notes": endpoint.notes,
        }

    except Exception as exc:
        latency_ms = round((time.perf_counter() - started) * 1000, 1)
        return {
            "category": endpoint.category, "endpoint": endpoint.name, "stability": endpoint.stability,
            "source_batch": endpoint.source_batch, "status": "REQUEST_EXCEPTION", "working": False,
            "http_status": None, "latency_ms": latency_ms, "result_count": 0, "examples": [],
            "examples_preview": "", "url": url, "error": f"{type(exc).__name__}: {exc}", "notes": endpoint.notes,
        }

def run_all(endpoints: list[Endpoint] = ENDPOINTS, query: str = QUERY) -> pd.DataFrame:
    rows = []
    total = len(endpoints)
    for idx, ep in enumerate(endpoints, start=1):
        print(f"[{idx:02d}/{total}] {ep.category} / {ep.name}")
        rows.append(test_endpoint(ep, query=query))
        if idx < total:
            time.sleep(DELAY_BETWEEN_REQUESTS_SECONDS)
    return pd.DataFrame(rows)


In [ ]:
# =========================
# RUN TESTS AND DISPLAY FINAL TABLE
# =========================

results_df = run_all(ENDPOINTS, query=QUERY)

final_cols = [
    "category", "endpoint", "stability", "source_batch", "status", "working",
    "http_status", "latency_ms", "result_count", "examples_preview", "error"
]

results_df[final_cols].sort_values(["category", "endpoint"]).reset_index(drop=True)


In [ ]:
# =========================
# SUMMARIES
# =========================

summary_by_status = results_df.groupby(["status"], dropna=False).size().reset_index(name="count").sort_values("count", ascending=False)
summary_by_category = results_df.groupby(["category", "status"], dropna=False).size().reset_index(name="count").sort_values(["category", "count"], ascending=[True, False])
summary_by_stability = results_df.groupby(["stability", "status"], dropna=False).size().reset_index(name="count").sort_values(["stability", "count"], ascending=[True, False])

display(summary_by_status)
display(summary_by_category)
display(summary_by_stability)


In [ ]:
# =========================
# WORKING ENDPOINTS ONLY
# =========================

working_df = results_df[results_df["working"] == True].copy()
working_df[["category", "endpoint", "stability", "source_batch", "http_status", "latency_ms", "result_count", "examples_preview", "url"]].reset_index(drop=True)


In [ ]:
# =========================
# FAILURES / SKIPS ONLY
# =========================

failures_df = results_df[results_df["working"] != True].copy()
failures_df[["category", "endpoint", "stability", "source_batch", "status", "http_status", "error", "url", "notes"]].reset_index(drop=True)


In [ ]:
# =========================
# DEDUPED EXAMPLE SUGGESTIONS
# =========================

all_suggestions = []
for _, row in results_df.iterrows():
    if not row.get("working"):
        continue
    for suggestion in row.get("examples", []):
        all_suggestions.append({
            "suggestion": suggestion,
            "endpoint": row["endpoint"],
            "category": row["category"],
            "stability": row["stability"],
            "source_batch": row["source_batch"],
        })

suggestions_df = pd.DataFrame(all_suggestions)
if not suggestions_df.empty:
    suggestions_df["suggestion_key"] = suggestions_df["suggestion"].str.lower().str.strip()
    deduped_suggestions_df = suggestions_df.drop_duplicates("suggestion_key").drop(columns=["suggestion_key"]).reset_index(drop=True)
else:
    deduped_suggestions_df = pd.DataFrame(columns=["suggestion", "endpoint", "category", "stability", "source_batch"])

deduped_suggestions_df


In [ ]:
# =========================
# EXPORT CSV / JSON / MARKDOWN
# =========================

run_stamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
safe_query = re.sub(r"[^a-zA-Z0-9_-]+", "_", QUERY).strip("_").lower()[:60] or "query"
out_dir = pathlib.Path(LOCAL_OUTPUT_DIR) / f"{safe_query}_{run_stamp}"
out_dir.mkdir(parents=True, exist_ok=True)

csv_path = out_dir / "keyword_endpoint_test_results.csv"
json_path = out_dir / "keyword_endpoint_test_results.json"
md_path = out_dir / "keyword_endpoint_test_results.md"
working_csv_path = out_dir / "working_keyword_endpoints.csv"
failures_csv_path = out_dir / "failed_or_skipped_keyword_endpoints.csv"
suggestions_csv_path = out_dir / "deduped_example_suggestions.csv"
new_entries_path = out_dir / "v2_new_entries_added.txt"

export_df = results_df.copy()
export_df["examples"] = export_df["examples"].apply(lambda xs: json.dumps(xs, ensure_ascii=False))
export_df.to_csv(csv_path, index=False)
results_df.to_json(json_path, orient="records", indent=2, force_ascii=False)
working_df.to_csv(working_csv_path, index=False)
failures_df.to_csv(failures_csv_path, index=False)
deduped_suggestions_df.to_csv(suggestions_csv_path, index=False)
new_entries_path.write_text("\n".join(NEW_ENTRIES_ADDED), encoding="utf-8")

markdown_table = results_df[final_cols].copy()
markdown_table["error"] = markdown_table["error"].fillna("").astype(str).str.slice(0, 240)
md_text = f"# Keyword Endpoint Test Results\n\nQuery: `{QUERY}`\n\nGenerated: `{dt.datetime.now().isoformat(timespec='seconds')}`\n\n"
md_text += "## New entries added in v2\n\n" + "\n".join([f"- {x}" for x in NEW_ENTRIES_ADDED])
md_text += "\n\n## Results\n\n" + markdown_table.to_markdown(index=False)
md_path.write_text(md_text, encoding="utf-8")

print("Saved local outputs:")
for p in [csv_path, json_path, md_path, working_csv_path, failures_csv_path, suggestions_csv_path, new_entries_path]:
    print(p)

if SAVE_TO_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        drive_dir = pathlib.Path(GOOGLE_DRIVE_OUTPUT_DIR) / f"{safe_query}_{run_stamp}"
        drive_dir.mkdir(parents=True, exist_ok=True)
        for p in [csv_path, json_path, md_path, working_csv_path, failures_csv_path, suggestions_csv_path, new_entries_path]:
            (drive_dir / p.name).write_bytes(p.read_bytes())
        print("\nSaved Google Drive outputs:")
        print(drive_dir)
    except Exception as exc:
        print(f"Google Drive save failed: {type(exc).__name__}: {exc}")


## Future endpoint template

```python
ENDPOINTS.append(
    Endpoint(
        category="General Web",
        name="Provider name",
        url_template="https://example.com/suggest?q={query}",
        parser=parse_generic,
        stability="experimental",
        source_batch="manual_addition",
    )
)
```
